# NHL Home Game Predictor — Evaluation & Sliding Window

Validate models on 2024-25 season with sliding window approach.

In [ ]:
import sys
sys.path.insert(0, '..')

import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import warnings
warnings.filterwarnings('ignore')

from pipeline.data_loader import load_all_seasons, TRAIN_SEASONS, TEST_SEASON
from pipeline import features, noise_reduction, calibration
from models import train, evaluate

df_all = load_all_seasons()
df_all_fe = features.build_features(df_all)

df_train_fe = noise_reduction.apply(df_all_fe[df_all_fe['season'].isin(TRAIN_SEASONS)].copy())
df_test_fe  = noise_reduction.apply(df_all_fe[df_all_fe['season'] == TEST_SEASON].copy())

print('Train:', df_train_fe.shape, '| Test:', df_test_fe.shape)

In [ ]:
# Train with N=20
train_feat_cols = evaluate.get_feature_cols_for_window(20, df_train_fe)
train_feat_cols = [c for c in train_feat_cols if c in df_train_fe.columns]

y_train = df_train_fe['result'].eq('W').astype(int)
X_train = df_train_fe[train_feat_cols].copy()
valid = X_train.notna().all(axis=1)
X_train, y_train = X_train[valid], y_train[valid]

models = train.train_all(X_train, y_train)

# Calibrate
cal_split = int(len(X_train) * 0.8)
X_cal, y_cal = X_train.iloc[cal_split:], y_train.iloc[cal_split:]
calibrated_models = {}
for name, model in models.items():
    try:
        calibrated_models[name] = calibration.calibrate(model, X_cal, y_cal)
    except Exception:
        calibrated_models[name] = model

In [ ]:
# Sliding window evaluation
results = evaluate.sliding_window_evaluation(calibrated_models, df_test_fe)
results

In [ ]:
# Best config table
pivot = evaluate.best_config_table(results)
print(pivot.to_string())

In [ ]:
# Accuracy vs N plot
baseline = 0.537
fig, ax = plt.subplots(figsize=(8, 5))
for model_name, grp in results.groupby('model'):
    ax.plot(grp['window'], grp['accuracy'], marker='o', label=model_name)
ax.axhline(baseline, color='k', linestyle='--', label=f'Baseline ({baseline})')
ax.set_xlabel('Window Size (N)')
ax.set_ylabel('Accuracy')
ax.set_title('Accuracy by Model & Window Size')
ax.legend()
plt.tight_layout()
plt.show()

In [ ]:
# Full metrics for best model
best_row = results.loc[results['accuracy'].idxmax()]
print(f"Best: {best_row['model']} | N={int(best_row['window'])}")
print(best_row[['accuracy','precision','recall','f1','brier','log_loss','roc_auc']].to_string())